# Get data

Builds the 2023-24 rolled player data from vaastav, football-data and Understat (`src/build_rolled_data.py`),
rebuilds the xP features without the gameweeks that have no target (`src/fix_missing_xp.py`), and writes one
file per position to `inputs/fg_data_{pos}.csv`.
Run `src/create_folds.py` (or `src/folds.sh`) afterwards to add the fold column.

In [6]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "../src")
import config  # noqa: E402  (its paths are relative to src/, which resolve the same from notebooks/)

In [7]:
# Build the season's data once (downloads are cached in inputs/sources/), then rebuild the xP features
if not Path(config.RAW_DATA).exists():
    subprocess.run([sys.executable, "build_rolled_data.py"], cwd="../src", check=True)
subprocess.run([sys.executable, "fix_missing_xp.py"], cwd="../src", check=True)
fg_data = pd.read_csv(config.RAW_DATA_XP_FIXED, low_memory=False)
fg_data

xP_rolling_1       max abs difference 0.00e+00
xP_1_ewm           max abs difference 0.00e+00
xP_rolling_3       max abs difference 0.00e+00
xP_3_ewm           max abs difference 0.00e+00
xP_rolling_5       max abs difference 0.00e+00
xP_5_ewm           max abs difference 0.00e+00
Rebuild matches the source data.
Set xP to empty in gameweeks [26]
Rows changed per feature: {'xP_rolling_1': 236, 'xP_1_ewm': 236, 'xP_rolling_3': 540, 'xP_3_ewm': 3242, 'xP_rolling_5': 914, 'xP_5_ewm': 3309}
Wrote ../inputs/rolled_player_data_23-24_xp_fixed.csv
Wrote ../inputs/rolled_player_data_23-24_all_players_xp_fixed.csv


,fpl_id,round,name,fpl_name,position,team,opponent_team,was_home,fixture,kickoff_time,...,xGChain_rolling_3,xGChain_3_ewm,xGChain_rolling_5,xGChain_5_ewm,xGBuildup_rolling_1,xGBuildup_1_ewm,xGBuildup_rolling_3,xGBuildup_3_ewm,xGBuildup_rolling_5,xGBuildup_5_ewm
0,2,24,Cédric Alves Soares,Cédric,DEF,Arsenal,West Ham,False,239,2024-02-11T14:00:00Z,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2,25,Cédric Alves Soares,Cédric,DEF,Arsenal,Burnley,False,242,2024-02-17T15:00:00Z,...,0.008179,0.012268,0.004907,0.008179,0.024536,0.024536,0.008179,0.012268,0.004907,0.008179
2,2,27,Cédric Alves Soares,Cédric,DEF,Arsenal,Sheffield Utd,False,269,2024-03-04T20:00:00Z,...,0.065585,0.046122,0.039351,0.041906,0.000000,0.000000,0.065585,0.046122,0.039351,0.041906
3,3,10,Mohamed Elneny,M.Elneny,MID,Arsenal,Sheffield Utd,True,91,2023-10-28T14:00:00Z,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,3,24,Mohamed Elneny,M.Elneny,MID,Arsenal,West Ham,False,239,2024-02-11T14:00:00Z,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11031,850,34,Ethan Wheatley,Wheatley,FWD,Man Utd,Sheffield Utd,True,288,2024-04-24T19:00:00Z,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
11032,850,36,Ethan Wheatley,Wheatley,FWD,Man Utd,Crystal Palace,False,356,2024-05-06T19:00:00Z,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
11033,850,37,Ethan Wheatley,Wheatley,FWD,Man Utd,Arsenal,True,365,2024-05-12T15:30:00Z,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
11034,854,38,Finley Munroe,Munroe,DEF,Aston Villa,Crystal Palace,False,376,2024-05-19T15:00:00Z,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [8]:
# Columns kept per position. Must include the numeric features listed in src/model_dispatcher.py.
GK_FEATS = [
    "fpl_id",
    "round",
    "position",
    "total_points",
    "was_home",
    "team",
    "opponent_team",
    "elo",
    "elo_h",
    "elo_a",
    "opp_elo",
    "opp_elo_h",
    "opp_elo_a",
    "team_h_difficulty",
    "team_a_difficulty",
    "win_prob",
    "value",
    "selected_by_percent",
    "transfers_in",
    "transfers_out",
    "total_points_rolling_3",
    "xGChain_rolling_3",
    "xGBuildup_rolling_3",
    "minutes_rolling_3",
    "starts_rolling_3",
    "expected_goals_conceded_rolling_3",
    "clean_sheets_rolling_3",
    "saves_rolling_3",
    "ict_index_rolling_3",
    "influence_rolling_3",
    "creativity_rolling_3",
    "threat_rolling_3",
]
DEF_FEATS = [
    "fpl_id",
    "round",
    "position",
    "total_points",
    "was_home",
    "team",
    "opponent_team",
    "elo",
    "elo_h",
    "elo_a",
    "opp_elo",
    "opp_elo_h",
    "opp_elo_a",
    "team_h_difficulty",
    "team_a_difficulty",
    "win_prob",
    "value",
    "selected_by_percent",
    "transfers_in",
    "transfers_out",
    "total_points_rolling_3",
    "xGChain_rolling_3",
    "xGBuildup_rolling_3",
    "expected_goals_rolling_3",
    "expected_assists_rolling_3",
    "expected_goal_involvements_rolling_3",
    "minutes_rolling_3",
    "starts_rolling_3",
    "expected_goals_conceded_rolling_3",
    "clean_sheets_rolling_3",
    "saves_rolling_3",
    "ict_index_rolling_3",
    "influence_rolling_3",
    "creativity_rolling_3",
    "threat_rolling_3",
]
MID_FEATS = [
    "fpl_id",
    "round",
    "position",
    "total_points",
    "was_home",
    "team",
    "opponent_team",
    "elo",
    "elo_h",
    "elo_a",
    "opp_elo",
    "opp_elo_h",
    "opp_elo_a",
    "team_h_difficulty",
    "team_a_difficulty",
    "win_prob",
    "value",
    "selected_by_percent",
    "transfers_in",
    "transfers_out",
    "total_points_rolling_3",
    "xGChain_rolling_3",
    "xGBuildup_rolling_3",
    "expected_goals_rolling_3",
    "expected_assists_rolling_3",
    "expected_goal_involvements_rolling_3",
    "minutes_rolling_3",
    "starts_rolling_3",
    "ict_index_rolling_3",
    "influence_rolling_3",
    "threat_rolling_3",
    "creativity_rolling_3",
]
FWD_FEATS = [
    "fpl_id",
    "round",
    "position",
    "total_points",
    "was_home",
    "team",
    "opponent_team",
    "elo",
    "elo_h",
    "elo_a",
    "opp_elo",
    "opp_elo_h",
    "opp_elo_a",
    "team_h_difficulty",
    "team_a_difficulty",
    "win_prob",
    "value",
    "selected_by_percent",
    "transfers_in",
    "transfers_out",
    "total_points_rolling_3",
    "xGChain_rolling_3",
    "xGBuildup_rolling_3",
    "expected_goals_rolling_3",
    "expected_assists_rolling_3",
    "expected_goal_involvements_rolling_3",
    "minutes_rolling_3",
    "starts_rolling_3",
    "ict_index_rolling_3",
    "influence_rolling_3",
    "threat_rolling_3",
    "creativity_rolling_3",
]

In [9]:
fg_data_gk = fg_data[fg_data["position"] == "GK"][GK_FEATS]
fg_data_def = fg_data[fg_data["position"] == "DEF"][DEF_FEATS]
fg_data_mid = fg_data[fg_data["position"] == "MID"][MID_FEATS]
fg_data_fwd = fg_data[fg_data["position"] == "FWD"][FWD_FEATS]

In [10]:
fg_data_gk.to_csv("../inputs/fg_data_gk.csv")
fg_data_def.to_csv("../inputs/fg_data_def.csv")
fg_data_mid.to_csv("../inputs/fg_data_mid.csv")
fg_data_fwd.to_csv("../inputs/fg_data_fwd.csv")


In [ ]:
import pandas as pd

fg_data_gk = pd.read_csv("../inputs/fg_data_gk.csv")